# APS360 Project: Data Pull Tests

Each section pulls a small sample from one data source and prints a **PASS** or **FAIL** line.
Only checks that each source works.

| Section | Source |
|---|---|
| 1 | S&P 500 prices (Yahoo Finance) |
| 2 | Fed target rate (FRED) |
| 3 | Kalshi Fed-rate markets |
| 4 | News headlines (NYT Archive API) |
| 5 | FinBERT sentiment |

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import sys, pandas as pd, numpy, requests, yfinance, torch, transformers, sklearn, dotenv
print(sys.version)
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("All packages loaded")

## 1. S&P 500 prices (Yahoo Finance)
Checks that daily prices download for the whole project period and computes the direction.

In [ ]:
import yfinance as yf
import numpy as np

spx = yf.download("^GSPC", start="2022-01-01", end="2026-10-01", progress=False, auto_adjust=False)
spx.columns = spx.columns.get_level_values(0)   # flatten column names
spx["log_return"] = np.log(spx["Close"] / spx["Close"].shift(1))
spx["label_up_tomorrow"] = (spx["Close"].shift(-1) > spx["Close"]).astype(int)

print(spx[["Close", "Volume", "log_return", "label_up_tomorrow"]].tail())
print("Trading days:", len(spx), "| first:", spx.index[0].date(), "| last:", spx.index[-1].date())
print("Fraction of up days:", round(spx["label_up_tomorrow"][:-1].mean(), 3))
print("PASS" if len(spx) > 1000 else "FAIL: fewer days than expected")

## 2. Fed target rate (FRED)
The upper bound of the federal funds target range, used to pick which Kalshi market to read each day.

In [ ]:
fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFEDTARU",
                   parse_dates=[0], index_col=0)
fred.columns = ["upper_bound"]
fred = fred.loc["2021-07-01":]

changes = fred[fred["upper_bound"].diff() != 0]
print(changes)   # every date the rate changed
print("PASS" if len(fred) > 0 else "FAIL")

## 3. Kalshi Fed-rate markets
Checks three things: the list of meetings, the markets for an **early (2022) meeting**, and **hourly** prices
for one market. Check if early markets have same "above X%" structure as recent ones.

In [ ]:
BASE = "https://api.elections.kalshi.com/trade-api/v2"

events = requests.get(f"{BASE}/events", params={"series_ticker": "KXFED", "limit": 200}).json()["events"]
print("Number of meetings:", len(events))
for e in sorted(events, key=lambda e: e.get("strike_date") or "")[:6]:
    print(" ", e["event_ticker"], "|", e["title"])

In [ ]:
def get_markets(event_ticker):
    """Settled markets come from the historical endpoint; open or recent ones from the live endpoint."""
    r = requests.get(f"{BASE}/historical/markets", params={"event_ticker": event_ticker, "limit": 100}).json()
    markets = r.get("markets", [])
    if not markets:
        r = requests.get(f"{BASE}/markets", params={"event_ticker": event_ticker, "limit": 100}).json()
        markets = r.get("markets", [])
    return markets

for event in ["FED-22DEC", "KXFED-26APR"]:
    markets = get_markets(event)
    print(f"\n{event}: {len(markets)} markets")
    for m in markets[:8]:
        print(f"  {m['ticker']:<22} {m.get('title','')[:70]} | volume {m.get('volume_fp')}")

In [ ]:
def hourly_candles(ticker, start, end):
    params = {"start_ts": int(pd.Timestamp(start, tz="UTC").timestamp()),
              "end_ts": int(pd.Timestamp(end, tz="UTC").timestamp()),
              "period_interval": 60}
    r = requests.get(f"{BASE}/historical/markets/{ticker}/candlesticks", params=params).json()
    rows = []
    for c in r.get("candlesticks", []):
        bid = float(c["yes_bid"]["close"]); ask = float(c["yes_ask"]["close"])
        rows.append({"time_et": pd.to_datetime(c["end_period_ts"], unit="s", utc=True).tz_convert("America/New_York"),
                     "bid": bid, "ask": ask, "mid": (bid + ask) / 2, "spread": ask - bid})
    return pd.DataFrame(rows)

# Change this to a ticker printed in the cell above (an early-2022 one is the most useful test).
TEST_TICKER = "KXFED-26APR-T4.25"
df = hourly_candles(TEST_TICKER, "2026-03-01", "2026-04-29")

if len(df):
    at_close = df[df["time_et"].dt.hour == 16]          # the 4 pm ET reading for each day
    print(at_close.tail(10))
    print(f"Hourly rows: {len(df)} | 4 pm rows: {len(at_close)} | "
          f"share of 4 pm rows with spread > 0.20: {(at_close['spread'] > 0.20).mean():.0%}")
    print("PASS")
else:
    print("FAIL: no candlesticks returned; check the ticker and dates")

## 4. News headlines (NYT Archive API)
Request returns one month.

In [ ]:
from dotenv import load_dotenv; import os
load_dotenv()
NYT_KEY = os.getenv("NYT_KEY")
print("Key loaded:", os.getenv("NYT_KEY") is not None)

def nyt_month(year, month):
    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    r = requests.get(url, params={"api-key": NYT_KEY}, timeout=120)
    r.raise_for_status()
    docs = r.json()["response"]["docs"]
    return pd.DataFrame([{
        "pub_date": d.get("pub_date"),
        "news_desk": d.get("news_desk"),
        "section": d.get("section_name"),
        "headline": (d.get("headline") or {}).get("main"),
        "abstract": d.get("abstract"),
    } for d in docs])

news = nyt_month(2024, 3)
news["pub_date"] = pd.to_datetime(news["pub_date"], utc=True).dt.tz_convert("America/New_York")
print("All articles that month:", len(news))
print(news["news_desk"].value_counts().head(10))   # check what the business desk is called

business = news[news["news_desk"].isin(["Business", "BusinessDay"])]
print("\nBusiness articles:", len(business),
      "| per day on average:", round(len(business) / news["pub_date"].dt.date.nunique(), 1))
print(business[["pub_date", "headline"]].head())
print("PASS" if len(business) > 0 else "FAIL: no business articles; check the news_desk names printed above")

## 5. FinBERT sentiment
Scores a few business headlines.

In [ ]:
from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)

def score(text):
    probs = {d["label"]: d["score"] for d in finbert(text[:512])[0]}
    return probs["positive"] - probs["negative"]

sample = business.head(10).copy()
sample["text"] = sample["headline"].fillna("") + ". " + sample["abstract"].fillna("")
sample["sentiment"] = sample["text"].apply(score)
print(sample[["headline", "sentiment"]].to_string())
print("PASS" if sample["sentiment"].between(-1, 1).all() else "FAIL")